# Phishing URL Detection

We train a Random Forest model on the phishing dataset and then use it to check if a given URL is safe or not.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import urllib.parse
import re
import itertools

## Load the Dataset

In [2]:
df = pd.read_csv('dataset_phishing.csv')
print("Shape:", df.shape)
print()
print("Label counts:")
print(df['status'].value_counts())
df.head()

Shape: (11430, 89)

Label counts:
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,...,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,...,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,...,0,1,0,224,8175,8725,0,0,6,legitimate


## Select Features and Train the Model

We use 50 URL-based features that can be extracted just by looking at the URL string itself.

In [3]:
# Features we can extract from the URL string
url_features = [
    'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 
    'nb_at', 'nb_qm', 'nb_and', 'nb_eq', 'nb_underscore', 'nb_tilde', 
    'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 
    'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 
    'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 
    'ratio_digits_host', 'punycode', 'port', 'tld_in_path',
    'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains',
    'prefix_suffix', 'random_domain', 'shortening_service',
    'path_extension', 'nb_redirection', 'nb_external_redirection',
    'length_words_raw', 'char_repeat', 'shortest_words_raw',
    'shortest_word_host', 'shortest_word_path', 'longest_words_raw',
    'longest_word_host', 'longest_word_path', 'avg_words_raw',
    'avg_word_host', 'avg_word_path', 'suspecious_tld'
]
url_features = [f for f in url_features if f in df.columns]

X = df[url_features]
y = df['status'].apply(lambda x: 1 if x == 'phishing' else 0)

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest model
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Check accuracy
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print()
print(classification_report(y_test, y_pred, target_names=['legitimate', 'phishing']))

Accuracy: 89.63%

              precision    recall  f1-score   support

  legitimate       0.89      0.91      0.90      1157
    phishing       0.90      0.88      0.89      1129

    accuracy                           0.90      2286
   macro avg       0.90      0.90      0.90      2286
weighted avg       0.90      0.90      0.90      2286



## Feature Importance

Which URL characteristics matter most?

In [4]:
importances = pd.Series(model.feature_importances_, index=url_features)
top = importances.sort_values(ascending=False).head(10)
print("Top 10 most important features:")
for feat, imp in top.items():
    print(f"  {feat}: {imp:.4f}")

Top 10 most important features:
  nb_www: 0.0829
  longest_word_path: 0.0598
  length_hostname: 0.0542
  length_url: 0.0541
  ratio_digits_url: 0.0533
  longest_words_raw: 0.0501
  char_repeat: 0.0488
  shortest_word_host: 0.0472
  nb_slash: 0.0455
  nb_hyphens: 0.0429


## Extract Features from Any URL

This function takes a raw URL string and computes all 50 features from it.

In [5]:
def extract_features(url):
    parsed = urllib.parse.urlparse(url)
    hostname = str(parsed.netloc) if parsed.netloc else ""
    path = str(parsed.path)
    parts = hostname.split('.')

    features = {}
    features['length_url'] = len(url)
    features['length_hostname'] = len(hostname)
    features['ip'] = 1 if re.match(r'^\d+\.\d+\.\d+\.\d+', hostname) else 0
    features['nb_dots'] = url.count('.')
    features['nb_hyphens'] = url.count('-')
    features['nb_at'] = url.count('@')
    features['nb_qm'] = url.count('?')
    features['nb_and'] = url.count('&')
    features['nb_eq'] = url.count('=')
    features['nb_underscore'] = url.count('_')
    features['nb_tilde'] = url.count('~')
    features['nb_percent'] = url.count('%')
    features['nb_slash'] = url.count('/')
    features['nb_star'] = url.count('*')
    features['nb_colon'] = url.count(':')
    features['nb_comma'] = url.count(',')
    features['nb_semicolumn'] = url.count(';')
    features['nb_dollar'] = url.count('$')
    features['nb_space'] = url.count(' ')
    features['nb_www'] = 1 if 'www.' in hostname else 0
    features['nb_com'] = 1 if '.com' in hostname else 0
    features['nb_dslash'] = url.count('//')
    features['http_in_path'] = 1 if 'http' in path else 0
    features['https_token'] = 1 if parsed.scheme == 'https' else 0
    digits_url = sum(c.isdigit() for c in url)
    features['ratio_digits_url'] = digits_url / len(url) if len(url) > 0 else 0
    digits_host = sum(c.isdigit() for c in hostname)
    features['ratio_digits_host'] = digits_host / len(hostname) if len(hostname) > 0 else 0
    features['punycode'] = 1 if 'xn--' in hostname else 0
    features['port'] = 1 if parsed.port and parsed.port not in [80, 443] else 0

    tld_list = ['.com', '.org', '.net', '.edu', '.gov', '.info', '.biz', '.co']
    features['tld_in_path'] = 1 if any(t in path for t in tld_list) else 0
    features['tld_in_subdomain'] = 0
    if len(parts) > 2:
        subdomain = '.'.join(parts[:-2])
        features['tld_in_subdomain'] = 1 if any(t in subdomain for t in tld_list) else 0
    features['abnormal_subdomain'] = 1 if len(parts) > 3 else 0
    if hostname.startswith('www.'):
        features['nb_subdomains'] = max(0, len(parts) - 2)
    else:
        features['nb_subdomains'] = max(0, len(parts) - 1)
    features['prefix_suffix'] = 1 if '-' in hostname else 0
    features['random_domain'] = 1 if re.search(r'[0-9a-f]{12,}', hostname.split('.')[0]) else 0
    features['shortening_service'] = 1 if any(s in hostname for s in ['bit.ly', 'tinyurl', 'goo.gl', 't.co', 'ow.ly']) else 0
    features['path_extension'] = 1 if re.search(r'\.(php|html|htm|asp|aspx|jsp|cgi)$', path) else 0
    features['nb_redirection'] = max(0, url.count('//') - 1)
    features['nb_external_redirection'] = 0

    words_raw = re.split(r'[\W_]+', url)
    words_raw = [w for w in words_raw if w]
    features['length_words_raw'] = len(words_raw)
    features['char_repeat'] = max((sum(1 for _ in g) for _, g in itertools.groupby(url)), default=0)
    if words_raw:
        features['shortest_words_raw'] = min(len(w) for w in words_raw)
        features['longest_words_raw'] = max(len(w) for w in words_raw)
        features['avg_words_raw'] = np.mean([len(w) for w in words_raw])
    else:
        features['shortest_words_raw'] = 0
        features['longest_words_raw'] = 0
        features['avg_words_raw'] = 0.0

    host_words = re.split(r'[\W_]+', hostname)
    host_words = [w for w in host_words if w]
    if host_words:
        features['shortest_word_host'] = min(len(w) for w in host_words)
        features['longest_word_host'] = max(len(w) for w in host_words)
        features['avg_word_host'] = np.mean([len(w) for w in host_words])
    else:
        features['shortest_word_host'] = 0
        features['longest_word_host'] = 0
        features['avg_word_host'] = 0.0

    path_words = re.split(r'[\W_]+', path)
    path_words = [w for w in path_words if w]
    if path_words:
        features['shortest_word_path'] = min(len(w) for w in path_words)
        features['longest_word_path'] = max(len(w) for w in path_words)
        features['avg_word_path'] = np.mean([len(w) for w in path_words])
    else:
        features['shortest_word_path'] = 0
        features['longest_word_path'] = 0
        features['avg_word_path'] = 0.0

    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.buzz', '.club']
    features['suspecious_tld'] = 1 if any(hostname.endswith(t) for t in suspicious_tlds) else 0

    return pd.DataFrame([features], columns=url_features)

## Check URLs

Run the cell below to test some sample URLs. Change them to whatever you want!

In [6]:
def check_url(url):
    features_df = extract_features(url)
    prediction = model.predict(features_df)[0]
    probability = model.predict_proba(features_df)[0]
    if prediction == 1:
        print(f"\u26a0\ufe0f  PHISHING: {url}  (confidence: {probability[1]*100:.1f}%)")
    else:
        print(f"\u2705 LEGITIMATE: {url}  (confidence: {probability[0]*100:.1f}%)")
    print()

check_url("https://www.google.com")
check_url("https://www.facebook.com/login")
check_url("http://secure-update-account-bank.com.login.verify.net/login.php")
check_url("http://192.168.1.1/admin/login.php")

✅ LEGITIMATE: https://www.google.com  (confidence: 80.5%)

✅ LEGITIMATE: https://www.facebook.com/login  (confidence: 60.5%)

⚠️  PHISHING: http://secure-update-account-bank.com.login.verify.net/login.php  (confidence: 86.0%)

⚠️  PHISHING: http://192.168.1.1/admin/login.php  (confidence: 92.5%)



## Try Your Own URL

In [ ]:
my_url = input("Enter a URL to check: ")
check_url(my_url)